# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library by referencing all data elements by their `@id`.

### Dataset Source
The dataset is described by a Croissant schema at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure required libraries are installed
!pip install -U mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load the dataset metadata and dataset records from the Croissant schema via `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Define schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Croissant metadata and dataset
dataset = mlc.Dataset(croissant_url)

# Display metadata summary
print("Dataset Metadata:")
print(f"Name: {dataset.metadata.name}")
print(f"Identifier: {getattr(dataset.metadata, 'identifier', None)}")
print(f"Version: {getattr(dataset.metadata, 'version', None)}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review record sets (`cr:RecordSet`), fields, and columns available in the dataset. Each element is identified by its `@id`.

> **Note:** As of writing, mlcroissant loads all record sets present in the schema or referenced as `recordSet` in the Croissant metadata.

In [ ]:
# Retrieve all record sets by @id, with their fields and columns
record_sets = dataset.metadata.record_sets

# Summarize available record sets and their fields/columns
overview = []

for rs in record_sets:
    rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', None)
    rs_name = getattr(rs, 'name', None)
    fields = getattr(rs, 'fields', [])
    field_ids = []
    for f in fields:
        field_ids.append(getattr(f, '@id', None) or getattr(f, 'id', None))
    columns = getattr(rs, 'columns', [])
    column_ids = []
    for c in columns:
        column_ids.append(getattr(c, '@id', None) or getattr(c, 'id', None))
    overview.append({
        'record_set': rs_id,
        'name': rs_name,
        'fields': field_ids,
        'columns': column_ids
    })

print("Record Sets Overview:")
for rs in overview:
    print(f"- Record Set: {rs['record_set']}")
    print(f"  Name: {rs['name']}")
    print(f"  Fields (@id): {rs['fields']}")
    print(f"  Columns (@id): {rs['columns']}\n")

## 3. Data Extraction
Load records from each available record set into a pandas DataFrame using their `@id` identifier.

We'll use the first record set for exploration. **All `@id`-s are referenced directly from the schema.**

In [ ]:
# Fetch all available record set @ids for extraction
record_set_ids = [rs['record_set'] for rs in overview]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

# Display basic info for the first loaded record set
if len(dataframes) > 0:
    first_rs_id = list(dataframes.keys())[0]
    print(f"First loaded record set: {first_rs_id}")
    print("Columns/Fields (@id):", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No records loaded. Please check record set IDs or dataset access.")

## 4. Exploratory Data Analysis (EDA)

Apply common data transformations and analyses, referencing all columns/fields by their exact `@id` as listed above.

We'll select a numeric field for filtering/normalization, and demonstrate group-based analysis. Replace `<numeric_field_id>` and `<group_field_id>` below according to discovered field/column `@id`s.

In [ ]:
# Choose the first record set for EDA
record_set_id = first_rs_id
df = dataframes[record_set_id]

# List columns for reference
print('Available columns for EDA:', df.columns.tolist())

# --- Select numeric field to analyze (by @id) ---
# Try to auto-detect candidates from column names
possible_numeric_fields = [c for c in df.columns if 'age' in c.lower() or df[c].dtype in [np.float64, np.int64, float, int]]
numeric_field_id = possible_numeric_fields[0] if possible_numeric_fields else df.columns[0]  # fallback
print(f"Selected numeric field for analysis: {numeric_field_id}")

# Filter records based on a numeric threshold (example: age > 50)
if np.issubdtype(df[numeric_field_id].dtype, np.number):
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (median):")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print(f"Field {numeric_field_id} is not numeric; skipping filtering/normalizing.")

# --- Grouping by a categorical field ---
# Try to pick a likely group field (contains 'sex', 'status', 'group', 'site' etc)
possible_group_fields = [c for c in df.columns if any(x in c.lower() for x in ['sex', 'status', 'site', 'location', 'msi'])]
group_field = possible_group_fields[0] if possible_group_fields else None

if group_field and group_field in filtered_df.columns:
    print(f"\nGrouping by {group_field}:")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(grouped_df)
else:
    print("No suitable group field found for group analysis.")

## 5. Visualization
Visualize distribution and relations of fields using referenced `@id`s. Example: histogram of the selected numeric field by group.

In [ ]:
# Visualize the distribution of the numeric field (possibly stratified by group, if available)
plt.figure(figsize=(8, 5))
if group_field and group_field in df.columns:
    sns.histplot(data=df, x=numeric_field_id, hue=group_field, bins=20, kde=True, palette="Set2", alpha=0.7)
    plt.title(f"Distribution of {numeric_field_id} by {group_field}")
else:
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")

plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to:
- Load the FAIR² dataset using its Croissant schema URL,
- Access and reference data entities by their stable `@id`,
- Extract, filter, normalize, group, and visualize data fields programmatically,
- Help ensure FAIR (Findable, Accessible, Interoperable, and Reusable) data practices.

Please refer to the dataset's Croissant schema for precise `@id` references of all entities. For more advanced analysis, repeat the above steps for any additional record sets or fields as needed.